In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Recuperamos dinámicamente las métricas que emitió la tarea de Silver
quarantine_rows = dbutils.jobs.taskValues.get(
    taskKey="02_transform_silver",
    key="quarantine_rows",
    default=0,
    debugValue=5  # Valor por defecto si lo pruebas manualmente fuera del Job
)

clean_rows = dbutils.jobs.taskValues.get(
    taskKey="02_transform_silver",
    key="clean_rows",
    default=0,
    debugValue=1000
)

total_rows = quarantine_rows + clean_rows
error_rate_pct = round((quarantine_rows / total_rows) * 100, 2) if total_rows > 0 else 0.0
severity = "CRITICAL" if error_rate_pct > 5.0 else "WARNING"

# Registramos el incidente en la tabla de observabilidad
audit_df = spark.createDataFrame(
    [(quarantine_rows, clean_rows, error_rate_pct, severity, "Corrupt sensor readings isolated in quarantine table")],
    ["quarantine_rows", "clean_rows", "error_rate_pct", "alert_severity", "incident_details"]
).withColumn("logged_at", current_timestamp())

audit_df.write.mode("append").saveAsTable("logistics_prod.gold.dq_audit_log")

print(f"🚨 ALERTA DE CALIDAD REGISTRADA [{severity}]: {quarantine_rows} filas en cuarentena ({error_rate_pct}%).")


